In [2]:
import os

files = os.listdir(r"C:\Users\juanb\Desktop\IronHack\vanguard_ab_test")
print(files)

['.git', 'df_final_demo.txt', 'df_final_experiment_clients.txt', 'df_final_web_data_pt_1.txt', 'df_final_web_data_pt_2.txt', 'dia 1 versión Juan.ipynb', 'Libro2.twb', 'Libro2.twbx', 'README.md', 'Test-AB-Vanguard.twbx', 'vanguard_tableau_clean.csv']


In [3]:
import pandas as pd

STEP_ORDER = ["start", "step_1", "step_2", "step_3", "confirm"]

df_pt1 = pd.read_csv(r"C:\Users\juanb\Desktop\IronHack\vanguard_ab_test\df_final_web_data_pt_1.txt")
df_pt2 = pd.read_csv(r"C:\Users\juanb\Desktop\IronHack\vanguard_ab_test\df_final_web_data_pt_2.txt")

df = pd.concat([df_pt1, df_pt2], ignore_index=True)

print(df.shape)
print(df.columns.tolist())
print(df.head())

(755405, 5)
['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time']
   client_id            visitor_id                      visit_id process_step  \
0    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   
1    9988021  580560515_7732621733  781255054_21935453173_531117       step_2   
2    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   
3    9988021  580560515_7732621733  781255054_21935453173_531117       step_2   
4    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   

             date_time  
0  2017-04-17 15:27:07  
1  2017-04-17 15:26:51  
2  2017-04-17 15:19:22  
3  2017-04-17 15:19:13  
4  2017-04-17 15:18:04  


In [5]:
df_demo = pd.read_csv(r"C:\Users\juanb\Desktop\IronHack\vanguard_ab_test\df_final_demo.txt")

print(df_demo.shape)
print(df_demo.columns.tolist())
print(df_demo.head())

(70609, 9)
['client_id', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'clnt_age', 'gendr', 'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth']
   client_id  clnt_tenure_yr  clnt_tenure_mnth  clnt_age gendr  num_accts  \
0     836976             6.0              73.0      60.5     U        2.0   
1    2304905             7.0              94.0      58.0     U        2.0   
2    1439522             5.0              64.0      32.0     U        2.0   
3    1562045            16.0             198.0      49.0     M        2.0   
4    5126305            12.0             145.0      33.0     F        2.0   

         bal  calls_6_mnth  logons_6_mnth  
0   45105.30           6.0            9.0  
1  110860.30           6.0            9.0  
2   52467.79           6.0            9.0  
3   67454.65           3.0            6.0  
4  103671.75           0.0            3.0  


In [6]:
df_experiment = pd.read_csv(r"C:\Users\juanb\Desktop\IronHack\vanguard_ab_test\df_final_experiment_clients.txt")

print(df_experiment.shape)
print(df_experiment.columns.tolist())
print(df_experiment.head())

(70609, 2)
['client_id', 'Variation']
   client_id Variation
0    9988021      Test
1    8320017      Test
2    4033851   Control
3    1982004      Test
4    9294070   Control


In [7]:
# ============================================================
# UNIR TODOS LOS DATAFRAMES
# ============================================================
df = pd.concat([df_pt1, df_pt2], ignore_index=True)
df = df.merge(df_experiment[["client_id", "Variation"]], on="client_id", how="left")

print(df.shape)
print(df["Variation"].value_counts())
print(df.head())

(755405, 6)
Variation
Test       177847
Control    143462
Name: count, dtype: int64
   client_id            visitor_id                      visit_id process_step  \
0    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   
1    9988021  580560515_7732621733  781255054_21935453173_531117       step_2   
2    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   
3    9988021  580560515_7732621733  781255054_21935453173_531117       step_2   
4    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   

             date_time Variation  
0  2017-04-17 15:27:07      Test  
1  2017-04-17 15:26:51      Test  
2  2017-04-17 15:19:22      Test  
3  2017-04-17 15:19:13      Test  
4  2017-04-17 15:18:04      Test  


In [8]:
import pandas as pd

STEP_ORDER = ["start", "step_1", "step_2", "step_3", "confirm"]

# ============================================================
# FUNCIÓN CONSOLIDATE STEPS
# ============================================================
def consolidate_steps(df, client_col="client_id", step_col="process_step"):
    dummies = pd.get_dummies(df[[client_col, step_col]], columns=[step_col])
    step_cols = [f"{step_col}_{s}" for s in STEP_ORDER if f"{step_col}_{s}" in dummies.columns]
    result = dummies.groupby(client_col)[step_cols].max().reset_index()
    result.columns = [client_col] + [c.replace(f"{step_col}_", "") for c in step_cols]
    df_cat = df.copy()
    df_cat[step_col] = pd.Categorical(df_cat[step_col], categories=STEP_ORDER, ordered=True)
    last_step = df_cat.groupby(client_col)[step_col].max().rename("last_step")
    result = result.merge(last_step, on=client_col)
    return result.reset_index(drop=True)

# ============================================================
# SEPARAR GRUPOS Y CONSOLIDAR
# ============================================================
df_control = df[df["Variation"] == "Control"].drop(columns="Variation")
df_test = df[df["Variation"] == "Test"].drop(columns="Variation")

result_control = consolidate_steps(df_control)
result_test = consolidate_steps(df_test)

step_counts_control = (
    df_control.groupby(["client_id", "process_step"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=STEP_ORDER, fill_value=0)
    .reset_index()
)

step_counts_test = (
    df_test.groupby(["client_id", "process_step"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=STEP_ORDER, fill_value=0)
    .reset_index()
)

print("Control:", result_control.shape, "Test:", result_test.shape)

Control: (23532, 7) Test: (26968, 7)


In [9]:
# ============================================================
# KPI 1 — COMPLETION RATE
# ============================================================
completion_rate_control = (result_control["last_step"] == "confirm").mean() * 100
completion_rate_test = (result_test["last_step"] == "confirm").mean() * 100

print(f"Completion rate (control): {completion_rate_control:.2f}%")
print(f"Completion rate (test):    {completion_rate_test:.2f}%")
print(f"Diferencia:                {completion_rate_test - completion_rate_control:.2f}pp")

Completion rate (control): 65.59%
Completion rate (test):    69.29%
Diferencia:                3.71pp


In [10]:
# ============================================================
# KPI 2 — TIME PER STEP
# ============================================================
def time_per_step(df_group):
    df_sorted = df_group.copy()
    df_sorted["date_time"] = pd.to_datetime(df_sorted["date_time"])
    df_sorted = df_sorted.sort_values(["client_id", "date_time"])
    df_sorted["time_next"] = df_sorted.groupby("client_id")["date_time"].shift(-1)
    df_sorted["time_spent"] = (
        (df_sorted["time_next"] - df_sorted["date_time"]).dt.total_seconds()
    )
    return (
        df_sorted.groupby("process_step")["time_spent"]
        .median()
        .reindex(STEP_ORDER)
        .reset_index()
    )

tps_control = time_per_step(df_control)
tps_control.columns = ["paso", "control_seg"]

tps_test = time_per_step(df_test)
tps_test.columns = ["paso", "test_seg"]

tps = tps_control.merge(tps_test, on="paso")
tps["diferencia_seg"] = tps["test_seg"] - tps["control_seg"]
print(tps)

      paso  control_seg  test_seg  diferencia_seg
0    start         22.0      16.0            -6.0
1   step_1         21.0      28.0             7.0
2   step_2         66.0      62.0            -4.0
3   step_3         75.0      59.0           -16.0
4  confirm        412.0     723.0           311.0


In [11]:
# ============================================================
# KPI 3 — ERROR RATE
# ============================================================
def error_rate(result_group, step_counts_group):
    completed = result_group[result_group["last_step"] == "confirm"][["client_id"]]
    completed_counts = completed.merge(step_counts_group, on="client_id")
    completed_counts["tuvo_error"] = (completed_counts[STEP_ORDER] > 1).any(axis=1)
    return completed_counts["tuvo_error"].mean() * 100

er_control = error_rate(result_control, step_counts_control)
er_test = error_rate(result_test, step_counts_test)

print(f"Error rate (control): {er_control:.2f}%")
print(f"Error rate (test):    {er_test:.2f}%")
print(f"Diferencia:           {er_test - er_control:.2f}pp")

Error rate (control): 53.42%
Error rate (test):    55.10%
Diferencia:           1.68pp


In [12]:
# ============================================================
# KPI 4 — MOMENTO DE ABANDONO
# ============================================================
def abandonment(result_group):
    abandoned = result_group[result_group["last_step"] != "confirm"]
    ab = (
        abandoned["last_step"]
        .value_counts()
        .reindex(STEP_ORDER[:-1])
        .reset_index()
    )
    ab.columns = ["paso_abandono", "clientes"]
    ab["pct_abandono_%"] = (ab["clientes"] / len(abandoned) * 100).round(2)
    return ab

ab_control = abandonment(result_control)
ab_control.columns = ["paso_abandono", "clientes_control", "pct_control_%"]

ab_test = abandonment(result_test)
ab_test.columns = ["paso_abandono", "clientes_test", "pct_test_%"]

ab = ab_control.merge(ab_test, on="paso_abandono")
ab["diferencia_pp"] = (ab["pct_test_%"] - ab["pct_control_%"]).round(2)
print(ab)

  paso_abandono  clientes_control  pct_control_%  clientes_test  pct_test_%  \
0         start              3285          40.57           2454       29.63   
1        step_1              1455          17.97           1980       23.91   
2        step_2              1265          15.62           1411       17.04   
3        step_3              2093          25.85           2436       29.42   

   diferencia_pp  
0         -10.94  
1           5.94  
2           1.42  
3           3.57  


In [13]:
funnel_control = result_control[STEP_ORDER].sum()
funnel_test = result_test[STEP_ORDER].sum()

dropoff_control = (1 - funnel_control / funnel_control.shift(1)) * 100
dropoff_test = (1 - funnel_test / funnel_test.shift(1)) * 100

dropoff = pd.DataFrame({
    "paso": STEP_ORDER,
    "dropoff_control_%": dropoff_control.values,
    "dropoff_test_%": dropoff_test.values
})
print(dropoff)

      paso  dropoff_control_%  dropoff_test_%
0    start                NaN             NaN
1   step_1          13.869299        9.040819
2   step_2           7.453355        8.278732
3   step_3           6.584450        6.186540
4  confirm          11.410860       10.507160


In [14]:
# Usuarios con más de una visita única
visits_per_client = df.groupby(["client_id", "Variation"])["visit_id"].nunique().reset_index()
visits_per_client.columns = ["client_id", "Variation", "num_visitas"]

return_rate = (
    visits_per_client.groupby("Variation")
    .apply(lambda x: (x["num_visitas"] > 1).mean() * 100)
    .reset_index()
)
return_rate.columns = ["Variation", "return_rate_%"]
print(return_rate)

  Variation  return_rate_%
0   Control      25.089240
1      Test      26.297834


In [15]:
df_sorted = df.copy()
df_sorted["date_time"] = pd.to_datetime(df_sorted["date_time"])

total_time = (
    df_sorted.groupby(["client_id", "Variation"])["date_time"]
    .agg(lambda x: (x.max() - x.min()).total_seconds())
    .reset_index()
)
total_time.columns = ["client_id", "Variation", "total_seg"]

# Solo completados
completed_ids = pd.concat([
    result_control[result_control["last_step"] == "confirm"][["client_id"]],
    result_test[result_test["last_step"] == "confirm"][["client_id"]]
])
total_time_completed = total_time[total_time["client_id"].isin(completed_ids["client_id"])]

print(total_time_completed.groupby("Variation")["total_seg"].median())

Variation
Control    378.0
Test       354.0
Name: total_seg, dtype: float64


# Análisis Test vs Control — Resumen
Lo que queríamos descubrir
Antes de meternos en los datos, nos hicimos tres preguntas:

H1 ✅ Spoiler: sí — ¿El nuevo diseño consigue que más gente termine el proceso?

H2 ✅ Spoiler: sí — ¿El nuevo diseño hace el proceso más rápido?

H3 ❌ Spoiler: no — ¿El nuevo diseño reduce los errores y los retrocesos?

## 1. Completion Rate → responde H1
Aquí las buenas noticias: un 3.71% más de usuarios termina el proceso con el nuevo diseño (65.59% → 69.29%). No es una diferencia brutal pero es real y consistente. H1 confirmada, aunque todavía hay que hacer el test estadístico para asegurarnos de que no es pura suerte.

## 2. Time per Step → responde H2
En general el nuevo diseño va más rápido, sobre todo en step_3 donde ahorra 16 segundos. El único dato raro es confirm, donde los usuarios del test tardan 311 segundos más que los del control. Probablemente dudan más justo antes de darle al botón final. Pero si miramos el tiempo total del proceso, el test sigue siendo más rápido (354s vs 378s), así que H2 también confirmada.

## 3. Error Rate → intenta responder H3, y falla
Aquí viene el jarro de agua fría. Esperábamos que el nuevo diseño redujera los errores pero la tasa es prácticamente igual en ambos grupos (53.42% vs 55.10%). O sea, más de la mitad de los usuarios que terminan el proceso se equivocan o retroceden en algún momento, da igual qué diseño estén viendo. H3 no confirmada: el rediseño no tocó los sitios donde la gente se lía.

## 4. Momento de Abandono → refuerza H1
El dato más llamativo de todo el análisis: el abandono en start cae un 10.94pp con el nuevo diseño. Eso explica directamente la mejora en completion rate de H1. El problema es que ese abandono no desaparece sino que se mueve a pasos posteriores, especialmente step_1 (+5.94pp) y step_3 (+3.57pp). El nuevo diseño engancha mejor al principio pero empuja el problema hacia adelante.

## 5. Drop-off Rate → refuerza H1 y matiza H2
Mismo mensaje: el nuevo diseño mejora mucho el salto de start a step_1 (-4.83pp) pero el resto de pasos quedan casi igual. El impacto del rediseño se nota sobre todo en la entrada, no tanto en el medio del flujo.

## 6. Tasa de Retorno → no aporta a ninguna hipótesis
Sin sorpresas aquí. En ambos grupos alrededor del 25% de los que abandonan vuelven a intentarlo. El nuevo diseño no cambia ese comportamiento para nada, lo que sugiere que la motivación para volver depende más del usuario que del diseño.

## 7. Tiempo Total → confirma H2
El test termina el proceso 24 segundos antes de mediana (354s vs 378s). Pequeño pero consistente con la mejora en step_3. H2 confirmada también por aquí.

## En resumen
El nuevo diseño cumple con 2 de las 3 cosas que queríamos mejorar: más gente termina el proceso y lo hace más rápido. Pero no resuelve el problema de los errores, que siguen por encima del 50% en ambos grupos. La mejora en completion rate (+3.71pp) es la estrella del análisis, pero antes de celebrar hay que confirmar que es estadísticamente significativa. Y una vez hecho eso, toca atacar step_1 y step_3, que siguen siendo los puntos donde más gente se pierde.



In [17]:
!pip install statsmodels

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.6 MB 1.2 MB/s eta 0:00:08
   --- ------------------------------------ 0.8/9.6 MB 1.2 MB/s eta 0:00:08
   ---- ----------------------------------- 1.0/9.6 MB 1.2 MB/s eta 0:00:08
   ----- ---------------------------------- 1.3/9.6 MB 1.2 MB/s eta 0:00:07
   ------ --------------------------------- 1.6/9.6 MB 1.2 MB/s eta 0:00:07
   ------- -------------------------------- 1.8/9.6 MB 1.2 MB/s eta 0:00:07
   -------- ------------------------------- 2.1/9.6 MB 1.2 MB/s eta 0:00:07
   --------- ------------------------------ 2.4/9.6 MB 1.2 MB/s eta 0:00:07
   ---------- ----------------------------- 2.6/9.6 MB 1.2 MB/s eta 0:00:06
   ------------ --------------------------- 2.9/9.6 MB 1.2 MB/s eta 0:00:06
   ------------- ----------------


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

# Datos por grupo
n_control = result_control.shape[0]
n_test = result_test.shape[0]

completados_control = (result_control["last_step"] == "confirm").sum()
completados_test = (result_test["last_step"] == "confirm").sum()

print(f"Control: {completados_control}/{n_control} ({completados_control/n_control*100:.2f}%)")
print(f"Test:    {completados_test}/{n_test} ({completados_test/n_test*100:.2f}%)")

# Z-test de proporciones
counts = np.array([completados_test, completados_control])
nobs = np.array([n_test, n_control])

stat, p_value = proportions_ztest(counts, nobs)

print(f"\nZ-statistic: {stat:.4f}")
print(f"P-value:     {p_value:.4f}")

# Conclusión
alpha = 0.05
if p_value < alpha:
    print(f"\n✅ Diferencia estadísticamente significativa (p < {alpha})")
else:
    print(f"\n❌ Diferencia NO estadísticamente significativa (p >= {alpha})")

Control: 15434/23532 (65.59%)
Test:    18687/26968 (69.29%)

Z-statistic: 8.8745
P-value:     0.0000

✅ Diferencia estadísticamente significativa (p < 0.05)


In [20]:
# Verificar si el aumento supera el umbral del 5%
diferencia = (completados_test/n_test) - (completados_control/n_control)
umbral = 0.05

print(f"Diferencia observada: {diferencia*100:.2f}pp")
print(f"Umbral requerido:     {umbral*100:.2f}pp")

if diferencia >= umbral:
    print(f"\n✅ La mejora ({diferencia*100:.2f}pp) supera el umbral del 5%")
else:
    print(f"\n❌ La mejora ({diferencia*100:.2f}pp) NO supera el umbral del 5%")

Diferencia observada: 3.71pp
Umbral requerido:     5.00pp

❌ La mejora (3.71pp) NO supera el umbral del 5%


Conclusión del test de hipótesis
La diferencia en completion rate entre Test y Control es estadísticamente significativa (p ≈ 0.0000, muy por debajo de 0.05), lo que significa que el resultado no es fruto del azar.
Sin embargo, la mejora observada de +3.71pp no supera el umbral mínimo del 5% exigido para considerar el nuevo diseño suficientemente efectivo.

# Abandono en el primer paso

In [21]:
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

# Clientes que abandonaron en start por grupo
abandono_start_control = (result_control["last_step"] == "start").sum()
abandono_start_test = (result_test["last_step"] == "start").sum()

n_control = result_control.shape[0]
n_test = result_test.shape[0]

print(f"Control: {abandono_start_control}/{n_control} ({abandono_start_control/n_control*100:.2f}%)")
print(f"Test:    {abandono_start_test}/{n_test} ({abandono_start_test/n_test*100:.2f}%)")

# Z-test de proporciones
counts = np.array([abandono_start_test, abandono_start_control])
nobs = np.array([n_test, n_control])

stat, p_value = proportions_ztest(counts, nobs)

print(f"\nZ-statistic: {stat:.4f}")
print(f"P-value:     {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"\n✅ Diferencia estadísticamente significativa (p < {alpha})")
else:
    print(f"\n❌ Diferencia NO estadísticamente significativa (p >= {alpha})")

Control: 3285/23532 (13.96%)
Test:    2454/26968 (9.10%)

Z-statistic: -17.1661
P-value:     0.0000

✅ Diferencia estadísticamente significativa (p < 0.05)


La diferencia en tasa de abandono en start entre Test y Control es estadísticamente significativa (p ≈ 0.0000). El nuevo diseño reduce el abandono en el primer paso del 13.96% al 9.10%, una caída de -4.86pp.
El Z-statistic negativo (-17.17) confirma que el Test tiene menos abandonos en start que el Control, y la magnitud del estadístico indica que es una diferencia muy robusta.


1. Efectividad del diseño

¿Cumple su objetivo principal? Parcialmente. El nuevo diseño mejora la completion rate (+3.71pp) de forma estadísticamente significativa, pero no alcanza el umbral mínimo del 5% establecido como criterio de éxito. Por tanto, no cumple el objetivo en su totalidad.

¿Qué funcionó bien?

Reducción drástica del abandono en start (-4.86pp, estadísticamente significativa)
Mejora del tiempo total del proceso (-24s de mediana)
Simplificación de step_3 (-16s)
Mayor engagement inicial: más usuarios entran al flujo
¿Qué no funcionó?

El abandono se redistribuye hacia step_1 (+5.94pp) y step_3 (+3.57pp)
La tasa de error no mejora (55.10% vs 53.42%)
El tiempo en confirm aumenta drásticamente (+311s)
La mejora en completion rate no supera el umbral del 5%
¿Recomendarías implementarlo? No de forma inmediata. El nuevo diseño muestra señales positivas pero no suficientes. La recomendación sería iterar sobre el diseño actual atacando específicamente step_1 y step_3, donde se concentra el abandono redistribuido, y reducir la fricción en confirm antes de un nuevo test.

2. Duración del experimento

In [22]:
# Rango de fechas del experimento
df["date_time"] = pd.to_datetime(df["date_time"])

print("Fecha inicio:", df["date_time"].min())
print("Fecha fin:   ", df["date_time"].max())
print("Duración:    ", (df["date_time"].max() - df["date_time"].min()).days, "días")

# Por grupo
print("\nControl:")
print("  Inicio:", df[df["Variation"]=="Control"]["date_time"].min())
print("  Fin:   ", df[df["Variation"]=="Control"]["date_time"].max())

print("\nTest:")
print("  Inicio:", df[df["Variation"]=="Test"]["date_time"].min())
print("  Fin:   ", df[df["Variation"]=="Test"]["date_time"].max())

Fecha inicio: 2017-03-15 00:03:03
Fecha fin:    2017-06-20 23:59:57
Duración:     97 días

Control:
  Inicio: 2017-03-15 00:19:28
  Fin:    2017-06-20 23:57:06

Test:
  Inicio: 2017-03-15 00:43:23
  Fin:    2017-06-20 23:21:23



El experimento duró 97 días (del 15 de marzo al 20 de junio de 2017), con ambos grupos corriendo exactamente en el mismo periodo, lo que es positivo para la validez del test.
¿Fue suficiente?
Sí. 97 días es una duración más que adecuada por varias razones:

Tamaño muestral: 23,532 usuarios en Control y 26,968 en Test, suficiente para detectar diferencias pequeñas con alta potencia estadística, como confirma el p-value prácticamente 0.
Cobertura temporal: casi 3 meses captura múltiples ciclos de comportamiento del usuario y elimina sesgos de días concretos o semanas atípicas.
Simultaneidad: ambos grupos corrieron en paralelo en exactamente el mismo periodo, eliminando sesgos estacionales.

Posibles problemas de duración:

97 días es un periodo largo que puede incluir cambios externos al experimento (campañas de marketing, cambios de mercado) que afecten los resultados.
No sabemos si hay efecto de novedad: los usuarios del Test podrían comportarse diferente al principio simplemente por ver algo nuevo, lo que inflaría artificialmente los resultados iniciales.

3. Datos adicionales necesarios
Basándonos en las limitaciones encontradas durante el análisis, estos son los datos que mejorarían el experimento:

Datos de comportamiento dentro del paso
Actualmente sabemos cuándo el usuario entra a cada paso pero no qué hace dentro. Necesitaríamos:

Clics en elementos específicos
Scroll depth
Campos del formulario que generan más abandonos

Esto explicaría por qué confirm tiene un tiempo tan alto (+311s) y dónde exactamente se pierden los usuarios en step_1 y step_3.

Datos de dispositivo y canal
No tenemos información sobre:

Si el usuario accede desde móvil o escritorio
Canal de entrada (email, directo, búsqueda)

El nuevo diseño podría funcionar mejor en un dispositivo que en otro, lo que explicaría la redistribución del abandono.

Datos de segmentación de usuarios
Aunque tenemos datos demográficos básicos, faltaría:

Si es la primera vez que el usuario intenta el proceso
Segmento de cliente (inversor nuevo vs experimentado)

Un usuario nuevo y uno experimentado probablemente reaccionan de forma muy distinta al nuevo diseño.

Datos post-experimento
No sabemos si los usuarios que completaron el proceso:

Quedaron satisfechos
Volvieron a usar el servicio
Tuvieron incidencias posteriores

Una tasa de completion alta no sirve de nada si los usuarios arrepienten o abandonan después.